# 실행 중인 그래프 200·400 실험 진행 보기 — CPU 읽기 전용

런타임 유형을 **CPU / 하드웨어 가속기 없음**으로 두고 Run all 하십시오. 기존 FAST 결과 폴더의 작은 진행 기록만 읽습니다. 30초마다 완료/전체·최근 속도·ETA·갱신 시각을 출력하며, GPU·모델·패키지 설치·실험 재실행·원본 파일 수정은 없습니다.

두 상한의 완료 기록이 해시까지 맞으면 자동 종료합니다. 원래 실행의 중단/실패나 기록 정체는 그대로 표시합니다. 첫 관측만으로 ETA를 만들지 않으며, 저장된 완료 문항 수가 늘어야 계산합니다. 모니터링 중단은 원래 GPU 작업에 영향을 주지 않습니다.


In [ ]:
from pathlib import Path
Path('/content/MUSTER_GRAPH_PROGRESS_READONLY.py').write_text('"""Read-only native FAST progress; no SDK, model, package install, or writes."""\nimport argparse as _lp_argparse\nfrom collections import deque as _lp_deque\nfrom datetime import datetime as _lp_datetime, timezone as _lp_timezone\nimport hashlib as _lp_hashlib\nimport json as _lp_json\nfrom pathlib import Path as _lp_Path\nimport threading as _lp_threading\nimport time as _lp_time\n\n\ndef _lp_read(path):\n    try:\n        raw = path.read_bytes()\n        value = _lp_json.loads(raw)\n        return (value, path.stat().st_mtime, raw) if isinstance(value, dict) else ({}, None, None)\n    except (OSError, ValueError):\n        return {}, None, None\n\n\ndef _lp_duration(seconds):\n    if seconds is None:\n        return "미정"\n    minutes, secs = divmod(max(0, int(seconds)), 60)\n    hours, minutes = divmod(minutes, 60)\n    return f"{hours:02d}:{minutes:02d}:{secs:02d}"\n\n\nclass LiveProgress:\n    """Print actual committed counts; ETA starts after a measured count delta."""\n\n    def __init__(self, root, *, cohort="lmev2_small451", total=451, caps=(200, 400),\n                 mirrored=False, interval=30, emit=print, clock=None):\n        self.root = _lp_Path(root)\n        self.cohort, self.total, self.caps = cohort, total, tuple(caps)\n        self.mirrored, self.interval, self.emit = mirrored, interval, emit\n        self.clock = clock or _lp_time.time\n        self.started = self.clock()\n        self.samples = {cap: _lp_deque() for cap in self.caps}\n        self.stop = _lp_threading.Event()\n        self.thread = None\n\n    def _job(self, cap):\n        phase = self.root / f"lmev2_cap{cap}"\n        return phase / ("files" if self.mirrored else "") / f"cap{cap}" / self.cohort\n\n    def sample(self):\n        now = self.clock()\n        rows = []\n        for cap in self.caps:\n            job = self._job(cap)\n            progress, updated, _ = _lp_read(job / "PROGRESS.json")\n            done, _, done_raw = _lp_read(job / "DONE.json")\n            fast, _, _ = _lp_read(job / "FAST_JOB_DONE.json")\n            failed, _, _ = _lp_read(job / "FAILED.json")\n            phase = self.root / f"lmev2_cap{cap}"\n            blocked, _, _ = _lp_read((phase / "files" if self.mirrored else phase) / "NATIVE_FAST_BLOCKED.json")\n            status, _, _ = _lp_read(self.root / f"lmev2_cap{cap}.STATUS.json")\n            count = progress.get("completed_queries")\n            valid_count = type(count) is int and 0 <= count <= self.total\n            if not valid_count:\n                count = None\n            complete = (fast.get("status") == "DONE" and done.get("complete") is True\n                        and done.get("questions") == self.total and done_raw is not None\n                        and fast.get("done_sha256") == _lp_hashlib.sha256(done_raw).hexdigest())\n            if complete:\n                count = self.total\n            samples = self.samples[cap]\n            reset = count is not None and bool(samples) and count < samples[-1][1]\n            if reset:\n                samples.clear()\n            if count is not None:\n                samples.append((now, count))\n                while len(samples) > 2 and samples[1][0] < now - 300:\n                    samples.popleft()\n            rate = eta = None\n            if len(samples) > 1:\n                dt, dc = now - samples[0][0], count - samples[0][1] if count is not None else 0\n                if dt >= 1 and dc > 0:\n                    rate = 60 * dc / dt\n                    eta = (self.total - count) / (rate / 60)\n            if complete:\n                state, eta = "완료기록 검증", 0\n            elif failed or blocked or status.get("status") in ("BLOCKED", "INTERRUPTED", "PARTIAL"):\n                state, eta, rate = "중단/실패 기록", None, None\n            elif count == self.total:\n                state, eta = "마지막 저장·완료 검증 중", None\n            elif count is not None:\n                state = "질문 기록 관측"\n            elif (job / "RUNTIME_HOOK.json").is_file():\n                state = "저장소 준비·첫 질문 대기"\n            elif (phase / ("files" if self.mirrored else "") / "NATIVE_FAST_CONFIG.json").is_file():\n                state = "검증·모델/저장소 준비"\n            else:\n                state = "시작 기록 대기"\n            age = max(0, now - updated) if updated is not None else None\n            if age is not None and age >= 180 and not complete:\n                state += " / 3분 이상 새 완료 기록 없음"\n            if reset:\n                state += " / 카운트 후퇴: ETA 재측정"\n            rows.append({"cap": cap, "completed": count, "total": self.total, "state": state,\n                         "complete": complete, "rate_q_per_min": rate, "eta_seconds": eta,\n                         "record_updated_epoch": updated, "record_age_seconds": age})\n        return {"utc": _lp_datetime.fromtimestamp(now, _lp_timezone.utc).strftime("%H:%M:%S UTC"),\n                "cohort": self.cohort, "watch_elapsed_seconds": now - self.started,\n                "questions_completed": sum(r["completed"] or 0 for r in rows),\n                "question_occurrences_total": self.total * len(self.caps), "caps": rows,\n                "all_complete": all(r["complete"] for r in rows),\n                "read_only": True, "model_API_calls": 0}\n\n    def report(self):\n        value = self.sample()\n        lines = [f"[{value[\'utc\']}] {self.cohort} | 전체 {value[\'questions_completed\']}/"\n                 f"{value[\'question_occurrences_total\']}문맥 | 모니터 경과 {_lp_duration(value[\'watch_elapsed_seconds\'])}"]\n        for row in value["caps"]:\n            count = "미관측" if row["completed"] is None else str(row["completed"])\n            speed = "미정" if row["rate_q_per_min"] is None else f"{row[\'rate_q_per_min\']:.2f}문항/분"\n            updated = "미관측" if row["record_updated_epoch"] is None else (\n                _lp_datetime.fromtimestamp(row["record_updated_epoch"], _lp_timezone.utc).strftime("%H:%M:%S UTC")\n                + f" ({int(row[\'record_age_seconds\'])}초 전)")\n            lines.append(f"  그래프 {row[\'cap\']} | {count}/{row[\'total\']} | {row[\'state\']} | "\n                         f"최근 속도 {speed} | ETA {_lp_duration(row[\'eta_seconds\'])} | 기록 갱신 {updated}")\n        lines.append("  완료 수는 저장된 질문 기록입니다. GPU 생존·정답률을 뜻하지 않습니다. ETA는 관측 후 남은 해당 상한의 추정입니다.")\n        self.emit("\\n".join(lines), flush=True)\n        return value\n\n    def watch(self):\n        while not self.stop.is_set():\n            if self.report()["all_complete"]:\n                return\n            self.stop.wait(self.interval)\n\n    def __enter__(self):\n        self.report()\n        def loop():\n            while not self.stop.wait(self.interval):\n                if self.report()["all_complete"]:\n                    break\n        self.thread = _lp_threading.Thread(target=loop, daemon=True)\n        self.thread.start()\n        return self\n\n    def __exit__(self, *_):\n        self.stop.set()\n        if self.thread:\n            self.thread.join(timeout=2)\n        self.report()\n\n\ndef progress_main():\n    parser = _lp_argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", default="/content/muster_graph_fast_work/cea67a239526543e")\n    parser.add_argument("--drive", action="store_true", help="read the existing Drive checkpoints")\n    parser.add_argument("--interval", type=float, default=30)\n    parser.add_argument("--once", action="store_true")\n    args = parser.parse_args()\n    if args.interval < 1:\n        parser.error("interval must be at least one second")\n    watcher = LiveProgress(args.root, mirrored=args.drive, interval=args.interval)\n    watcher.report() if args.once else watcher.watch()\n\n\nif __name__ == "__main__":\n    progress_main()\n', encoding='utf-8')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib.util
spec = importlib.util.spec_from_file_location('_readonly_progress', '/content/MUSTER_GRAPH_PROGRESS_READONLY.py')
progress = importlib.util.module_from_spec(spec)
spec.loader.exec_module(progress)
watch = progress.LiveProgress('/content/drive/MyDrive/MUSTER_graph_budget_fast_20261116/results/cea67a239526543e', mirrored=True, interval=30)
watch.watch()
